# Bias-Variance Tradeoff Laboratory Report

## Question 1: Decision Tree Depth vs. Overfitting

In [1]:
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Generate synthetic binary classification dataset
X, y = make_classification(n_samples=500, n_features=10, random_state=42)

# Split dataset into 70% training and 30% testing subsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Instantiate an underfit decision tree restricted to depth 1
model_underfit = DecisionTreeClassifier(max_depth=1, random_state=42)
# Fit the underfit decision stump on training data
model_underfit.fit(X_train, y_train)

# 2. Instantiate an unconstrained overfit decision tree with no depth limit
model_overfit = DecisionTreeClassifier(max_depth=None, random_state=42)
# Fit the unconstrained tree on training data
model_overfit.fit(X_train, y_train)

# 3. Compute predictions and accuracies for the underfit model
train_acc_underfit = accuracy_score(y_train, model_underfit.predict(X_train))
test_acc_underfit = accuracy_score(y_test, model_underfit.predict(X_test))

# Compute predictions and accuracies for the overfit model
train_acc_overfit = accuracy_score(y_train, model_overfit.predict(X_train))
test_acc_overfit = accuracy_score(y_test, model_overfit.predict(X_test))

# Print performance metrics
print(f"Underfit Model (max_depth=1)    -> Train Acc: {train_acc_underfit:.4f}, Test Acc: {test_acc_underfit:.4f}")
print(f"Overfit Model  (max_depth=None) -> Train Acc: {train_acc_overfit:.4f}, Test Acc: {test_acc_overfit:.4f}")

Underfit Model (max_depth=1)    -> Train Acc: 0.8914, Test Acc: 0.8800
Overfit Model  (max_depth=None) -> Train Acc: 1.0000, Test Acc: 0.9000


### Logical Questions:

#### 1. Based on your printed accuracies, which model demonstrates High Bias and which demonstrates High Variance?
* **Model Demonstrating High Bias:**  
  The Decision Tree with `max_depth=1` exhibits **High Bias (Underfitting)**. Restricting tree depth to a single decision stump limits model capacity, resulting in lower and tightly bounded accuracies on both training (**0.8914**) and testing (**0.8800**) sets.
* **Model Demonstrating High Variance:**  
  The unconstrained Decision Tree with `max_depth=None` exhibits **High Variance (Overfitting)**. It achieves a perfect **100% training accuracy** (**1.0000**) but drops to **90.00%** (**0.9000**) on the test set. The notable 10% performance gap between training and testing sets serves as empirical evidence of high variance.

#### 2. Why does the unconstrained tree (max_depth=None) achieve 100% accuracy on the training set but perform worse on the test set?
* An unconstrained tree (`max_depth=None`) recursively splits nodes until every leaf node achieves zero Gini impurity.
* Consequently, the tree memorizes sample-specific noise and exact feature patterns in the training data.
* When evaluating unseen test samples, these memorized rules fail to generalize perfectly, leading to the observed drop from 100% training accuracy to 90% test accuracy.

## Question 2: Ridge Regularization (L2) and Variance Reduction

In [2]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error

# Generate noisy high-dimensional data
np.random.seed(0)
X_train = np.random.randn(20, 50)  # 20 samples, 50 features (High variance setup)
y_train = np.random.randn(20)
X_test = np.random.randn(20, 50)
y_test = np.random.randn(20)

# 1. Instantiate and fit LinearRegression() as lr_model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 2. Instantiate and fit Ridge(alpha=100.0) as ridge_model
ridge_model = Ridge(alpha=100.0, random_state=0)
ridge_model.fit(X_train, y_train)

# 3. Calculate and print Test MSE for both models using mean_squared_error
y_pred_lr = lr_model.predict(X_test)
y_pred_ridge = ridge_model.predict(X_test)

test_mse_lr = mean_squared_error(y_test, y_pred_lr)
test_mse_ridge = mean_squared_error(y_test, y_pred_ridge)

print(f"Linear Regression Test MSE: {test_mse_lr:.4f}")
print(f"Ridge Regression  Test MSE: {test_mse_ridge:.4f}")

Linear Regression Test MSE: 1.6844
Ridge Regression  Test MSE: 0.8960


### Logical Questions:

#### 1. Why does standard LinearRegression suffer from high variance when the number of features (50) exceeds the number of samples (20)?
* When feature dimension ($p = 50$) exceeds sample size ($n = 20$), the system of linear equations is underdetermined, making the covariance matrix $X^T X$ singular and non-invertible.
* Standard Ordinary Least Squares (OLS) fits sample-specific noise, causing coefficient weight vector magnitudes to inflate.
* Consequently, minute shifts in input features produce higher output fluctuations, leading to an elevated Test Mean Squared Error (MSE = **1.6844**) for unregularized Linear Regression.

#### 2. How does increasing alpha in Ridge regression impact the Bias and Variance of the model?
* Ridge regression minimizes the L2-regularized objective:
  $$\mathcal{L}_{\text{Ridge}}(\mathbf{w}) = \|\mathbf{y} - X\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_2^2$$
* **Impact on Variance:** Increasing $\alpha$ adds positive regularization terms along the matrix diagonal ($(X^T X + \alpha I)^{-1}$), shrinking parameter weights toward zero. This stabilizes matrix inversion and heavily suppresses sensitivity to training noise, leading to a **significant reduction in Variance** (Test MSE drops from **1.6844** down to **0.8960**).
* **Impact on Bias:** By constraining coefficient magnitudes, Ridge restricts overall hypothesis space capacity, which **increases Bias**.
* The trade-off is optimal because the substantial variance reduction far outweighs the minor bias increase, resulting in superior test generalization.

## Question 3: Polynomial Feature Expansion and Curve Fitting

In [3]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Fix random seed for reproducibility
np.random.seed(42)

# Generate noisy quadratic training samples
X_train = np.linspace(-2, 2, 15).reshape(-1, 1)
y_train = X_train.ravel()**2 + np.random.normal(0, 0.5, 15)

# Generate noisy quadratic testing samples
X_test = np.linspace(-2, 2, 10).reshape(-1, 1)
y_test = X_test.ravel()**2 + np.random.normal(0, 0.5, 10)

def evaluate_polynomial(degree):
    # 1. Transform X_train and X_test using PolynomialFeatures(degree)
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    # 2. Fit LinearRegression on transformed training data
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    # Predict outputs for train and test sets
    y_pred_train = model.predict(X_train_poly)
    y_pred_test = model.predict(X_test_poly)
    
    # Compute train and test Mean Squared Errors
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    
    # 3. Return (train_mse, test_mse)
    return train_mse, test_mse

# Run for Degree 1 (Linear) and Degree 10 (High Polynomial)
print("Degree 1: ", evaluate_polynomial(1))
print("Degree 10:", evaluate_polynomial(10))

Degree 1:  (1.4266798575254636, 1.9972064157941485)
Degree 10: (0.060397682104422964, 0.3251365609052635)


### Logical Questions:
#### 1. Degree 1 fits a straight line to quadratic data. Does Degree 1 fail due to high bias or high variance?
* Degree 1 fits a straight linear line ($y = w_0 + w_1 x$) to data generated by a quadratic parabolic function ($y \approx x^2$).
* A linear hypothesis space lacks the functional capacity to represent quadratic curvature.
* Therefore, Degree 1 fails strictly due to **High Bias (Underfitting)**, producing elevated error metrics on both training (**1.4267**) and testing (**1.9972**) sets.

#### 2. Why does Degree 10 produce a much higher Test MSE than Degree 1, despite having a lower Train MSE?
* A 10th-degree polynomial has 11 trainable parameters to fit only 15 training points.
* It achieves a substantially lower Train MSE (**0.0604**) compared to Degree 1 because its excessive degrees of freedom allow the flexible curve to pass directly through individual noisy training points.
* Although Degree 10 captures the general parabolic trend better than a flat line in this small sample setup (resulting in a lower Test MSE of **0.3251** compared to Degree 1), it exhibits **High Variance (Overfitting)** relative to its training fit, as demonstrated by the relative error jump from training (**0.0604**) to testing (**0.3251**).

## Question 4: Impact of Dataset Size on Model Variance

In [4]:
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Dataset setup
X, y = make_classification(n_samples=4000, n_features=20, random_state=42)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=1000, random_state=42
)

# Subsets
X_train_small, y_train_small = X_train_full[:30], y_train_full[:30]
X_train_large, y_train_large = X_train_full[:3000], y_train_full[:3000]

# 1. Fit RandomForestClassifier(random_state=42) on (X_train_small, y_train_small)
rf_small = RandomForestClassifier(n_estimators=100, random_state=42)
rf_small.fit(X_train_small, y_train_small)

# 2. Fit RandomForestClassifier(random_state=42) on (X_train_large, y_train_large)
rf_large = RandomForestClassifier(n_estimators=100, random_state=42)
rf_large.fit(X_train_large, y_train_large)

# 3. Print the Test Accuracy on X_test for both models
acc_small = accuracy_score(y_test, rf_small.predict(X_test))
acc_large = accuracy_score(y_test, rf_large.predict(X_test))

print(f"Random Forest (30 samples)   -> Test Accuracy: {acc_small:.4f}")
print(f"Random Forest (3000 samples) -> Test Accuracy: {acc_large:.4f}")

Random Forest (30 samples)   -> Test Accuracy: 0.9160
Random Forest (3000 samples) -> Test Accuracy: 0.9510


### Logical Questions:
#### 1. How does increasing the amount of training data affect a model's Variance?
* Increasing the volume of training data **significantly reduces Model Variance**.
* When trained on a tiny sample size ($n = 30$), a high-capacity model (Random Forest) fits sample-specific noise and accidental patterns, resulting in higher variance and a lower test accuracy (**0.9160**).
* As sample size expands to $n = 3000$, empirical data statistics converge toward the true underlying population distribution by the Law of Large Numbers. Individual sample noise averages out, stabilizing decision trees and raising Test Accuracy to **0.9510**.

#### 2. If a model suffers from High Bias, will adding more training data solve the problem? Explain why or why not based on model capacity.
* **No, adding more training data will NOT solve a High Bias problem.**
* High Bias (Underfitting) occurs when a model lacks sufficient **hypothesis capacity** or makes oversimplified assumptions about the target function (e.g., fitting a linear line to quadratic data).
* Once the training set is large enough to estimate optimal parameters for that constrained model, adding extra samples provides no benefit—the model fundamentally cannot represent complex decision boundaries.
* As demonstrated in learning curve theory, both training and validation error curves plateau at an unacceptably high error baseline when High Bias is present. Resolving High Bias requires increasing model capacity (e.g., deeper trees, non-linear kernels) rather than collecting more data.